In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\processed\amazon_sales_cleaned.csv')

In [3]:
df.head()

,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,qty,amount,ship_state,b2b,fulfilled_by
0,2022-04-30,cancelled,merchant,amazon.in,standard,set389,set389_kr_np_s,set,s,0,647.62,maharashtra,False,easy_ship
1,2022-04-30,shipped_delivered_to_buyer,merchant,amazon.in,standard,jne3781,jne3781_kr_xxxl,kurta,3xl,1,406.00,karnataka,False,easy_ship
2,2022-04-30,shipped,amazon,amazon.in,expedited,jne3371,jne3371_kr_xl,kurta,xl,1,329.00,maharashtra,True,NaN
3,2022-04-30,cancelled,merchant,amazon.in,standard,j0341,j0341_dr_l,western_dress,l,0,753.33,tamil_nadu,False,easy_ship
4,2022-04-30,shipped,amazon,amazon.in,expedited,jne3671,jne3671_tu_xxxl,top,3xl,1,574.00,tamil_nadu,False,NaN


In [4]:
df['is_cancelled'] = df['status'] == 'cancelled'

fulfilment_stats = df.groupby('fulfilment').agg(
    total_orders=('status', 'count'),
    cancelled_orders=('is_cancelled', 'sum'),
    total_revenue=('amount', 'sum'),
    lost_revenue=('amount', lambda x: x[df.loc[x.index, 'is_cancelled']].sum())
).reset_index()

fulfilment_stats['cancellation_rate'] = (fulfilment_stats['cancelled_orders'] / fulfilment_stats['total_orders'] * 100).round(2)
fulfilment_stats['lost_revenue_pct'] = (fulfilment_stats['lost_revenue'] / fulfilment_stats['total_revenue'] * 100).round(2)

print("CANCELLATION BY FULFILLMENT TYPE")
print("=" * 50)
print("\nBy Fulfilment Channel (Amazon vs Merchant):")
print(fulfilment_stats.to_string(index=False))

fulfilled_by_stats = df.groupby('fulfilled_by', dropna=False).agg(
    total_orders=('status', 'count'),
    cancelled_orders=('is_cancelled', 'sum')
).reset_index()

fulfilled_by_stats['fulfilled_by'] = fulfilled_by_stats['fulfilled_by'].fillna('Not Specified')
fulfilled_by_stats['cancellation_rate'] = (fulfilled_by_stats['cancelled_orders'] / fulfilled_by_stats['total_orders'] * 100).round(2)

print("\n\nBy Fulfilled By Method:")
print(fulfilled_by_stats.to_string(index=False))

CANCELLATION BY FULFILLMENT TYPE

By Fulfilment Channel (Amazon vs Merchant):
fulfilment  total_orders  cancelled_orders  total_revenue  lost_revenue  cancellation_rate  lost_revenue_pct
    amazon         87613             10863    53136483.00    3505589.00              12.40              6.60
  merchant         38628              6519    23902157.19    3032929.19              16.88             12.69


By Fulfilled By Method:
 fulfilled_by  total_orders  cancelled_orders  cancellation_rate
    easy_ship         38628              6519              16.88
Not Specified         87613             10863              12.40


In [6]:
fulfilment_stats.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\cancellation_by_fulfillment.csv', index=False)
fulfilled_by_stats.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\cancellation_by_fulfilled_by.csv', index=False)
print("Saved to data/Queried/")

Saved to data/Queried/


In [5]:
revenue_by_state = df.groupby('ship_state').agg(
    total_revenue=('amount', 'sum'),
    total_orders=('status', 'count')
).reset_index()

revenue_by_state['pct_of_total_revenue'] = (revenue_by_state['total_revenue'] / revenue_by_state['total_revenue'].sum() * 100).round(2)
revenue_by_state = revenue_by_state.sort_values('total_revenue', ascending=False).reset_index(drop=True)

print("REVENUE BY STATE")
print("=" * 50)
print(revenue_by_state.to_string(index=False))

REVENUE BY STATE
       ship_state  total_revenue  total_orders  pct_of_total_revenue
      maharashtra    13057892.19         21786                 16.95
        karnataka    10070373.99         16650                 13.08
    uttar_pradesh     6736403.84         10484                  8.75
        telangana     6653694.04         10900                  8.64
       tamil_nadu     6563135.62         11553                  8.52
            delhi     4270631.48          6845                  5.54
           kerala     3791206.64          6500                  4.92
      west_bengal     3462444.61          5865                  4.50
   andhra_pradesh     3192221.46          5359                  4.14
          haryana     2859667.12          4377                  3.71
          gujarat     2709392.00          4455                  3.52
        rajasthan     1748541.23          2693                  2.27
   madhya_pradesh     1585911.74          2508                  2.06
           punjab

In [6]:
revenue_by_state.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\revenue_by_state.csv', index=False)
print("Saved to data/Queried/")

Saved to data/Queried/


In [7]:
revenue_by_cat_size = df.groupby(['category', 'size']).agg(
    total_revenue=('amount', 'sum'),
    total_orders=('status', 'count')
).reset_index()

revenue_by_cat_size['pct_of_total_revenue'] = (revenue_by_cat_size['total_revenue'] / revenue_by_cat_size['total_revenue'].sum() * 100).round(2)
revenue_by_cat_size = revenue_by_cat_size.sort_values('total_revenue', ascending=False).reset_index(drop=True)

print("REVENUE BY CATEGORY AND SIZE")
print("=" * 50)
print(revenue_by_cat_size.to_string(index=False))

REVENUE BY CATEGORY AND SIZE
     category size  total_revenue  total_orders  pct_of_total_revenue
          set    m     7138117.52          9113                  9.27
          set    l     6236139.50          8022                  8.09
          set   xl     5804911.29          7400                  7.54
          set    s     5714410.91          7321                  7.42
          set  xxl     4710966.76          6088                  6.12
          set  3xl     4476110.16          5714                  5.81
          set   xs     4197227.36          5395                  5.45
        kurta    l     3652977.07          8715                  4.74
        kurta   xl     3560351.47          8478                  4.62
        kurta    m     3478771.30          8387                  4.52
        kurta  xxl     3173170.75          7525                  4.12
        kurta  3xl     2416520.43          5584                  3.14
        kurta    s     2315484.64          5635              

In [8]:
revenue_by_category = df.groupby('category').agg(
    total_revenue=('amount', 'sum'),
    total_orders=('status', 'count')
).reset_index()

revenue_by_category['pct_of_total_revenue'] = (revenue_by_category['total_revenue'] / revenue_by_category['total_revenue'].sum() * 100).round(2)
revenue_by_category = revenue_by_category.sort_values('total_revenue', ascending=False).reset_index(drop=True)

print("REVENUE BY CATEGORY")
print("=" * 50)
print(revenue_by_category.to_string(index=False))

REVENUE BY CATEGORY
     category  total_revenue  total_orders  pct_of_total_revenue
          set    38488207.81         49237                 49.96
        kurta    20919922.57         48901                 27.16
western_dress    10845319.86         14971                 14.08
          top     5269679.90         10461                  6.84
 ethnic_dress      787337.37          1153                  1.02
       blouse      452655.94           914                  0.59
       bottom      150667.98           437                  0.20
        saree      123933.76           164                  0.16
      dupatta         915.00             3                  0.00


In [9]:
revenue_by_category.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\revenue_by_category.csv', index=False)
revenue_by_cat_size.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\revenue_by_category_size.csv', index=False)
print("Saved to data/Queried/")

Saved to data/Queried/


In [10]:
shipping_completion = df.groupby(['ship_service_level', 'status']).agg(
    total_orders=('status', 'count'),
    total_revenue=('amount', 'sum')
).reset_index()

shipping_totals = df.groupby('ship_service_level')['status'].count().reset_index(name='group_total')
shipping_completion = shipping_completion.merge(shipping_totals, on='ship_service_level')
shipping_completion['pct_of_group'] = (shipping_completion['total_orders'] / shipping_completion['group_total'] * 100).round(2)

print("SHIPPING SPEED AND ORDER COMPLETION")
print("=" * 50)
print(shipping_completion.to_string(index=False))

SHIPPING SPEED AND ORDER COMPLETION
ship_service_level                      status  total_orders  total_revenue  group_total  pct_of_group
         expedited                   cancelled         10815     3503227.00        86531         12.50
         expedited                     pending           407      264521.00        86531          0.47
         expedited                     shipped         75309    49331767.00        86531         87.03
          standard                   cancelled          6567     3035291.19        39710         16.54
          standard                     pending           239      158160.00        39710          0.60
          standard pending_waiting_for_pick_up           276      188199.00        39710          0.70
          standard                     shipped          1024       34606.00        39710          2.58
          standard             shipped_damaged             1        1136.00        39710          0.00
          standard  shipped_delivered

In [11]:
shipping_completion.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\shipping_speed_completion.csv', index=False)
print("Saved to data/Queried/")

Saved to data/Queried/


In [12]:
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').astype(str)

monthly_revenue = df.groupby('month').agg(
    total_revenue=('amount', 'sum'),
    total_orders=('status', 'count'),
    avg_order_value=('amount', 'mean')
).reset_index()

monthly_revenue['avg_order_value'] = monthly_revenue['avg_order_value'].round(2)
monthly_revenue['revenue_change_pct'] = monthly_revenue['total_revenue'].pct_change().mul(100).round(2)

print("MONTHLY REVENUE TRENDS")
print("=" * 50)
print(monthly_revenue.to_string(index=False))

MONTHLY REVENUE TRENDS
  month  total_revenue  total_orders  avg_order_value  revenue_change_pct
2022-03      101035.85           170           627.55                 NaN
2022-04    28194601.31         47905           625.71            27805.54
2022-05    25691334.20         41111           663.34               -8.88
2022-06    23051668.83         37055           660.94              -10.27


In [13]:
monthly_revenue.to_csv(r'C:\Documents\Data analytics\projects\my-data-project\E-COMMERCE-PROJECT\data\Queried\monthly_revenue_trends.csv', index=False)
print("Saved to data/Queried/")

Saved to data/Queried/
